## Example Ring and Sidechain Datasets

In [3]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from pathlib import Path

# Define the directory where the files will be saved
HOME_DIR = Path.cwd()  # Change this if needed
DATA_DIR = HOME_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)  # Ensure the directory exists

# Create Rings Dataset
rings_data = [
    {"id": "R1", "smiles": "A", "n_subs": 2},  # Benzene with 2 attachment points
    {"id": "R2", "smiles": "B", "n_subs": 3},  # Pyridine with 3 attachment points
    {"id": "R3", "smiles": "C", "n_subs": 1},  # Pyridine with 1 attachment point
]

# Convert to DataFrame
df_rings = pd.DataFrame(rings_data)

# Save as Parquet
df_rings.to_parquet(DATA_DIR / "ring_list.parquet", engine="pyarrow")

# Create Sidechains Dataset
sidechains_data = [
    {"id": "S1", "smiles": "E"},   # Methoxy (-OCH3)
    {"id": "S2", "smiles": "F"},  # Chloromethyl (-CH2Cl)
]

# Convert to DataFrame
df_sidechains = pd.DataFrame(sidechains_data)

# Save as Parquet
df_sidechains.to_parquet(DATA_DIR / "sidechain_list.parquet", engine="pyarrow")

print(f"✅ Rings and Sidechains datasets saved in {DATA_DIR}")


✅ Rings and Sidechains datasets saved in C:\Users\zachg\PycharmProjects\BillionMolecules\billion_mol_db\testing_recombination\data


## Recombination Logic

In [21]:
import duckdb
import pandas as pd
import concurrent.futures
import pyarrow as pa
from pathlib import Path
import pyarrow.parquet as pq
import itertools
import os
import shutil


def new_combination(ring_smiles, sub_smiles_list):
    """Generates a new combined SMILES string from a ring and multiple sidechains."""
    return ring_smiles + "(" + ")(".join(sub_smiles_list) + ")"


# Function to process and write each batch to its own file
def process_and_write_batch(df_batch, parquet_schema, batch_id, temp_dir):
    df_batch["combination"] = df_batch.apply(
        lambda row: new_combination(row["ring_smiles"], [row[f"sub{i}_smiles"] for i in range(1, row["n_subs"] + 1)]),
        axis=1,
    )

    df_batch = df_batch.drop(columns=["n_subs"])  # Remove n_subs column after processing

    # Convert DataFrame to Apache Arrow table
    table = pa.Table.from_pandas(df_batch, schema=parquet_schema)

    # Write to a temporary Parquet file
    temp_file = os.path.join(temp_dir, f"batch_{batch_id}.parquet")
    pq.write_table(table, temp_file)


def main():
    # Variable and file paths
    HOME_DIR = Path.cwd()  # Gets the current working directory
    # HOME_DIR = Path(__file__).resolve().parent.parent
    ring_file = HOME_DIR / "data" / "ring_list.parquet"
    sidechain_file = HOME_DIR / "data" / "sidechain_list.parquet"
    mol_db_file = HOME_DIR / "data" / "mol_db.parquet"
    temp_dir = HOME_DIR / "temp_batches"  # Directory to store temporary batch files
    temp_dir.mkdir(exist_ok=True)

    id_name = "id"
    smiles_name = "smiles"
    n_subs_name = "n_subs"
    batch_size = 1000  # Adjust this batch size as necessary

    # Load the molecule fragments from a file into DuckDB
    con = duckdb.connect()
    con.execute(f"CREATE TABLE rings AS SELECT * FROM '{ring_file}'")  # Load rings into DuckDB
    con.execute(f"CREATE TABLE sidechains AS SELECT * FROM '{sidechain_file}'")  # Load sidechains into DuckDB

    # Set up Parquet schema dynamically based on the maximum number of substitutions
    max_subs = con.execute(f"SELECT MAX({n_subs_name}) FROM rings").fetchone()[0]
    parquet_schema_fields = [
        ("ring_id", pa.string()),
        ("ring_smiles", pa.string()),
        ("n_subs", pa.int32()),  # Keep track of how many substitutions
    ]
    for i in range(1, max_subs + 1):
        parquet_schema_fields.append((f"sub{i}_id", pa.string()))
        parquet_schema_fields.append((f"sub{i}_smiles", pa.string()))
    parquet_schema_fields.append(("combination", pa.string()))  # Final combined SMILES
    parquet_schema = pa.schema(parquet_schema_fields)

    # Create the base query for rings and the first sidechain (must be an INNER JOIN to guarantee at least one sidechain)
    query = f"""
        SELECT rings.{id_name} AS ring_id, rings.{smiles_name} AS ring_smiles, rings.{n_subs_name} AS n_subs,
               side1.{id_name} AS sub1_id, side1.{smiles_name} AS sub1_smiles
        FROM rings
        INNER JOIN sidechains AS side1 ON TRUE
        WHERE rings.{n_subs_name} >= 1
        """
    
    # Dynamically add additional LEFT JOINs for more sidechains based on `n_subs`
    for i in range(2, max_subs + 1):
        query += f"""
        LEFT JOIN sidechains AS side{i} ON rings.{n_subs_name} >= {i}
        """
    
    print("Generated SQL Query:\n", query)  # Debugging: Check query before execution

    # Execute the query and use fetch_df_chunk to fetch data in chunks
    cursor = con.execute(query)

    executor = concurrent.futures.ProcessPoolExecutor()
    batch_id = 0

    while True:
        # Fetch the next chunk of data using fetch_df_chunk()
        df_batch = cursor.fetch_df_chunk(batch_size)
        if df_batch.empty:
            break  # No more data

        # Process each batch in parallel and write to its own file
        future = executor.submit(
            process_and_write_batch, df_batch=df_batch, parquet_schema=parquet_schema, batch_id=batch_id, temp_dir=temp_dir
        )

        batch_id += 1

    # After processing, merge all temp files into the final file
    with pq.ParquetWriter(mol_db_file, parquet_schema) as writer:
        for temp_file in temp_dir.glob("*.parquet"):
            table = pq.read_table(temp_file)
            writer.write_table(table)

    total_rows = con.execute(f"SELECT COUNT(*) FROM ({query})").fetchone()[0]
    print(f"Mol DB with {total_rows} rows written to {mol_db_file}.")

    # Cleanup temporary files and connections
    shutil.rmtree(temp_dir)  # Remove entire temp directory
    con.close()
    executor.shutdown()


if __name__ == "__main__":
    main()


Generated SQL Query:
 
        SELECT rings.id AS ring_id, rings.smiles AS ring_smiles, rings.n_subs AS n_subs,
               side1.id AS sub1_id, side1.smiles AS sub1_smiles
        FROM rings
        INNER JOIN sidechains AS side1 ON TRUE
        WHERE rings.n_subs >= 1
        
        LEFT JOIN sidechains AS side2 ON rings.n_subs >= 2
        
        LEFT JOIN sidechains AS side3 ON rings.n_subs >= 3
        


ParserException: Parser Error: syntax error at or near "LEFT"

In [12]:
import networkx
import rdkit

    atom in_ring
0      0      no
1      1      no
2      2      no
3      3      no
4      4      no
5      5      no
6      6     yes
7      7     yes
8      8     yes
9      9     yes
10    10     yes
11    11     yes
12    12     yes
13    13     yes
14    14     yes
15    15     yes
16    16     yes
17    17     yes
18    18     yes
19    19     yes
20    20     yes
21    21     yes
22    22     yes
23    23     yes
24    24     yes
25    25     yes
26    26     yes
27    27     yes
28    28      no
29    29      no
30    30      no
31    31      no
32    32      no
33    33      no
34    34      no
35    35      no
36    36      no
37    37      no
38    38      no
39    39      no
40    40      no
41    41      no
42    42      no
43    43      no
44    44      no
45    45      no
